In [0]:
BASE_DIR = '/Volumes/sales/landing/volumes/Customers/'
CHECKPOINT_DIR = '/Volumes/sales/landing/volumes/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/customers'
CATALOG = 'SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'CUSTOMERS'
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:
customers_schema = """customer_id STRING, 
                 customer_unique_id STRING, 
                 customer_zip_code_prefix INT, 
                 customer_city STRING, 
                 customer_state STRING
                 """

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

cust_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(customers_schema)
                 .load(f'{BASE_DIR}')
        )

cust_trans_df = cust_df.select(
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

In [0]:
qry = (
    cust_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()